<a href="https://colab.research.google.com/github/Ruthless3217/Qwen-Fine-Tuned-Model/blob/main/qwen_compliance_colab_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦙 Qwen 2.5 – Compliance Fine-Tuning (Google Colab)

**Before running:** Make sure your runtime is set to **T4 GPU**.
Go to **Runtime → Change runtime type → T4 GPU** then click Save.

---
## Step 1: Install Dependencies

In [1]:
%%capture
# Install Unsloth (Colab-specific build) and its dependencies
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl datasets transformers accelerate bitsandbytes

## Step 2: Upload & Extract Your Dataset

You need to zip your `dataset_2.1_rl` folder and upload it. Run the cell below to do it interactively.

In [2]:
from google.colab import files
import os

print("Please upload your dataset_2.1_rl.zip file now:")
uploaded = files.upload()

# Get the uploaded filename
if not uploaded:
    print("❌ No file uploaded!")
else:
    zip_filename = list(uploaded.keys())[0]
    print(f"Uploaded: {zip_filename}")

    # Unzip the dataset
    !unzip -q "{zip_filename}" -d .

    # Verify it extracted correctly
    # We assume the zip contains a folder named 'dataset_2.1_rl' or files directly
    dataset_folder = "dataset_2.1_rl"
    if os.path.isdir(dataset_folder):
        file_count = len([f for f in os.listdir(dataset_folder) if f.endswith('.json')])
        print(f"✅ Dataset extracted! Found {file_count} JSON files in '{dataset_folder}'")
    else:
        print(f"⚠️ Could not find folder '{dataset_folder}'. Checking top-level...")
        file_count = len([f for f in os.listdir('.') if f.endswith('.json')])
        if file_count > 0:
            print(f"✅ Found {file_count} JSON files in current directory. Setting DATA_DIR to '.'")
            DATA_DIR = "."
        else:
            print("❌ No JSON files found. Please check the zip structure.")

Please upload your dataset_2.1_rl.zip file now:


Saving dataset_2.1_rl.zip to dataset_2.1_rl (1).zip
Uploaded: dataset_2.1_rl (1).zip
replace ./dataset_2.1_rl/10398_10398_10398_7._Mortality_Charges_in_ULIP_Investment.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
✅ Dataset extracted! Found 2617 JSON files in 'dataset_2.1_rl'


## Step 3: Configuration & Hyperparameters

In [3]:
import os

# --- Dataset folder ---
if 'DATA_DIR' not in locals():
    DATA_DIR = "dataset_2.1_rl"

# --- Model ---
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Fits perfectly in T4 (16GB VRAM)
                                          # For 9B, switch to "Qwen/Qwen2.5-9B-Instruct"

OUTPUT_DIR = "./compliance_qwen_model"

# --- Training Hyperparameters ---
MAX_SEQ_LENGTH = 4096   # T4 has 16GB so 4096 should be fine; lower to 2048 if OOM
LOAD_IN_4BIT   = True   # 4-bit QLoRA: essential for fitting large models in 16GB
BATCH_SIZE     = 2      # Lower to 1 and increase GRADIENT_ACCUMULATION to 8 if OOM
GRADIENT_ACCUMULATION = 4
EPOCHS         = 3
LEARNING_RATE  = 2e-4

print("✅ Configuration set.")
print(f"   Model      : {MODEL_NAME}")
print(f"   Data dir   : {DATA_DIR}")
print(f"   Seq length : {MAX_SEQ_LENGTH}")
print(f"   4-bit      : {LOAD_IN_4BIT}")
print(f"   Batch size : {BATCH_SIZE}")

✅ Configuration set.
   Model      : Qwen/Qwen2.5-3B-Instruct
   Data dir   : dataset_2.1_rl
   Seq length : 4096
   4-bit      : True
   Batch size : 2


## Step 4: Load & Format the Dataset

In [4]:
import json
from datasets import Dataset

def load_and_format_dataset(data_dir):
    """
    Reads all JSON files from `data_dir`.
    Each file must have keys: 'instruction', 'input', and 'output' (with 'final_text' inside).
    Returns a HuggingFace Dataset formatted as chat conversations.
    """
    conversations = []

    for filename in os.listdir(data_dir):
        if not filename.endswith(".json"):
            continue

        filepath = os.path.join(data_dir, filename)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)

            instruction = data.get("instruction", "")
            user_input  = data.get("input", "")
            final_text  = data.get("output", {}).get("final_text", "")

            if not instruction or not final_text:
                continue

            convo = [
                {"role": "system",    "content": "You are a compliance-focused insurance documentation assistant."},
                {"role": "user",      "content": f"{instruction}\n\n{user_input}"},
                {"role": "assistant", "content": final_text}
            ]
            conversations.append({"messages": convo})

        except Exception as e:
            print(f"⚠️ Error loading {filepath}: {e}")

    hf_dataset = Dataset.from_list(conversations)
    print(f"✅ Loaded {len(hf_dataset)} training examples.")
    return hf_dataset

dataset = load_and_format_dataset(DATA_DIR)
print(dataset)

✅ Loaded 2617 training examples.
Dataset({
    features: ['messages'],
    num_rows: 2617
})


## Step 5: Load Model & Tokenizer with Unsloth

In [5]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print("Loading model and tokenizer... (this may take a few minutes)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype         = None,       # Auto-detect: bf16 on A100/T4, fp16 otherwise
    load_in_4bit  = LOAD_IN_4BIT,
)

# Apply the Qwen-specific chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Changed from "qwen2" to "chatml"
)

print(f"✅ Model loaded: {MODEL_NAME}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model and tokenizer... (this may take a few minutes)
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


✅ Model loaded: Qwen/Qwen2.5-3B-Instruct


## Step 6: Apply Chat Template to Dataset

In [6]:
def apply_chat_template(examples):
    """Formats messages into the Qwen chat prompt format."""
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(apply_chat_template, batched=True)
print("✅ Chat template applied to dataset.")

# Preview a single example
print("\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]["text"][:500])

Map:   0%|          | 0/2617 [00:00<?, ? examples/s]

✅ Chat template applied to dataset.

--- Sample formatted text (first 500 chars) ---
<|im_start|>system
You are a compliance-focused insurance documentation assistant.<|im_end|>
<|im_start|>user
Rewrite the content to meet insurance regulatory and compliance standards.

{'draft_text': 'Reasons you need Term Insurance if you are Self Employed\nMeta description: Do self-employed individuals still need term insurance? Here are 5 reasons why they should invest in one immediately.\nKeywords: term insurance, what is term insurance, term insurance benefits, online term plan\nA term ins


## Step 7: Apply LoRA Adapters

In [7]:
print("Applying LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    lora_alpha         = 16,
    lora_dropout       = 0,           # Must be 0 for Unsloth optimized training
    bias               = "none",      # Must be 'none' for Unsloth optimized training
    use_gradient_checkpointing = "unsloth",  # Saves ~30% VRAM
    random_state       = 3407,
)

print("✅ LoRA adapters applied.")

Applying LoRA adapters...


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ LoRA adapters applied.


## Step 8: Train the Model

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Initializing SFTTrainer...")

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = dataset,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing         = True,   # Packs multiple short examples into one sequence for efficiency
    args = TrainingArguments(
        per_device_train_batch_size   = BATCH_SIZE,
        gradient_accumulation_steps   = GRADIENT_ACCUMULATION,
        warmup_steps                  = 100,
        num_train_epochs              = EPOCHS,
        learning_rate                 = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps                 = 10,
        optim                         = "adamw_8bit",  # 8-bit optimizer saves VRAM
        weight_decay                  = 0.01,
        lr_scheduler_type             = "linear",
        seed                          = 3407,
        output_dir                    = "lora_checkpoints",
        report_to                     = "none",         # Disable wandb unless you want it
    ),
)

print("Starting training... (grab a coffee ☕)")
trainer_stats = trainer.train()

print("\n✅ Training complete!")
print(f"   Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Final loss: {trainer_stats.metrics['train_loss']:.4f}")

Initializing SFTTrainer...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2617 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training... (grab a coffee ☕)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,617 | Num Epochs = 3 | Total steps = 984
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
10,2.157160
20,2.110242
30,1.935578
40,1.865658
50,1.846076
60,1.811378
70,1.725308
80,1.708476
90,1.769089
100,1.726092



✅ Training complete!
   Runtime: 7509s
   Final loss: 1.5262


## Step 9: Save the Fine-Tuned Model (LoRA Adapters)

In [9]:
print(f"Saving LoRA adapters to '{OUTPUT_DIR}'...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA adapters saved to '{OUTPUT_DIR}'")

Saving LoRA adapters to './compliance_qwen_model'...
✅ LoRA adapters saved to './compliance_qwen_model'


## Step 10: (Optional) Save Fully Merged 16-bit Model

This merges LoRA weights into the base model. Produces a larger file but is standalone — no need for PEFT at inference time.

⚠️ This requires ~18GB of free disk space in Colab.

In [10]:
merged_dir = f"{OUTPUT_DIR}_merged"
print(f"Saving merged 16-bit model to '{merged_dir}'...")
model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
print(f"✅ Merged model saved to '{merged_dir}'")

Saving merged 16-bit model to './compliance_qwen_model_merged'...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:42<00:00, 21.12s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:32<00:00, 46.37s/it]


Unsloth: Merge process complete. Saved to `/content/compliance_qwen_model_merged`
✅ Merged model saved to './compliance_qwen_model_merged'


## Step 11: (Optional) Mount Google Drive to Save Model Permanently

In [11]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

# --- Copy to Drive ---
drive_save_path = '/content/drive/MyDrive/compliance_qwen_model'
print(f"Copying model to {drive_save_path}...")
if os.path.exists(drive_save_path):
  shutil.rmtree(drive_save_path)
shutil.copytree(OUTPUT_DIR, drive_save_path)
print("✅ Model saved to Google Drive!")

MessageError: Error: credential propagation was unsuccessful

---
## Step 12: Run Inference with the Fine-Tuned Model

Test the model immediately after training — no need to reload unless your session reset.

In [13]:
FastLanguageModel.for_inference(model)

# --- Change this to any insurance article you want to rewrite ---
insurance_article = """
If you crash your car, we will pay you money. You need to call us within 2 days or we won't pay.
Also, don't lie to us because if we find out, we keep the money.
"""

messages = [
    {"role": "system",  "content": "You are a compliance-focused insurance documentation assistant."},
    {"role": "user",    "content": f"Rewrite the content to meet insurance regulatory and compliance standards.\n\n{insurance_article}"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize              = True,
    add_generation_prompt = True,
    return_tensors        = "pt",
).to("cuda")

print("Generating compliant rewrite...")
outputs = model.generate(
    input_ids    = inputs['input_ids'], # Correctly access the input_ids tensor
    max_new_tokens = 1024,
    use_cache    = True,
    temperature  = 0.3,
    top_p        = 0.9,
)

# Decode and print only the newly generated tokens (skip the prompt)
# Correctly access the input_ids tensor's shape
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("\n--- REWRITTEN COMPLIANT ARTICLE ---")
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generating compliant rewrite...

--- REWRITTEN COMPLIANT ARTICLE ---
This statement is not accurate or relevant to insurance regulations. Here’s why:

1. **Car Insurance Claim Process:**
   If you have an accident, you must inform your insurer as soon as possible. However, there is no rule that says you must report within two days. The claim process can take some time, but it depends on the type of insurance policy and the insurer's internal procedures.

2. **Liability for Delayed Reporting:**
   There is no provision in insurance laws that states you won’t get paid if you report the incident after two days. In most cases, insurers allow a reasonable timeframe for reporting accidents. The exact period varies from one company to another, but it is usually a few days rather than just two days.

3. **No Requirement to Inform Others:**
   Your insurer does not have the right to ask you to inform other parties about the accident. It is your responsibility to report the incident to your insu